# Run Unlearning Experiments

This notebook allows the user to set varius configs for a particular unlearning scenario, runs the protocols, measures results, and pulls in the checkpoints and results for the relevant original and retrain-from-scratch models.

### Imports

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt


from data.utils import split_forget_retain, split_random
from data.dataloaders import unmark_dataset
import time
from unlearn.utils import do_unlearning
from trainer.utils import init_folder_if_not_exists

### Set configs for the experiment

In [2]:

from master_hyperparams import hyperparams

device = "cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu"

# ---- main configs for this experiment ----- #

dataset = "CIFAR10"
model_class = "ResNet"
unlearning_type = "class"
reference_methods = ["FT", "GA", "NegGrad_plus", "RL", "boundary_shrink", "bad_teacher", "scrub", "UNSIR"]
measure_base_results = True
measure_retrain_results = True
num_runs = 3

# ------------------------------------------- #

hp = hyperparams[dataset]
model_hp = hp[model_class]

exp_config = {

    "description": "First official run - ResNet18, CIFAR10, class unlearning",
    
    "device": device,
    "model_class": model_class,
    "unlearning_type": unlearning_type,
    "num_runs": num_runs,
    "measure_base_results": measure_base_results,
    "measure_retrain_results": measure_retrain_results,

    "data": {
        "dataset": dataset,
        "num_classes": hp["num_classes"],
        "batch_size": hp["batch_size"],
        "num_workers": hp["num_workers"],
        "item_to_unlearn": hp["items_to_unlearn"][unlearning_type]
        },

    "training": model_hp["training"],
    
    "unlearning": {
        "methods": reference_methods,
        **model_hp["unlearning"]
        }
}


### Protocol for several runs

In [3]:
import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /cs/student/msc/ml/2025/jmoncus/.netrc.
wandb: Currently logged in as: jjmoncus (jjmoncus706) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
import glob
from models.archs.utils import init_model
from torch.optim.lr_scheduler import ReduceLROnPlateau
from trainer.utils import training_regimen_lr_annealing
from data.dataloaders import load_dataloaders_for_experiment
from evaluation.utils import measure_solo_metrics, measure_solo_and_comparison_metrics
import json
from data.utils import setup_seed

def run_experiment(config, results_folder, checkpoint_folder):
    
    print("="*70)
    print("="*19 + "  " + f'RUNNING EXPERIMENT, SEED {config["GRAND_SEED"]}' + "  " + "="*19)
    print("="*70 + "\n")

    setup_seed(config["GRAND_SEED"])

    # Make experiment results folder if it doesnt already exist
    if not os.path.exists(results_folder):
        print(f"{results_folder} doesn't exist - creating it...\n")
        os.makedirs(results_folder, exist_ok=True)

    # Save the config for this experiment to the main results folder
    with open(os.path.join(results_folder, "experiment_config.json"), "w") as f:
        json.dump(config, f, indent=4)

    # create a subfolder for saving model checkpoints for this experiment
    print(f'All models will be of class {config["model_class"]}.\n')
    checkpoint_subfolder = os.path.join(checkpoint_folder, f"seed_{config['GRAND_SEED']}")
    if not os.path.exists(checkpoint_subfolder):   
        print(f"{checkpoint_subfolder} doesn't exist - creating it...\n")
        os.makedirs(checkpoint_subfolder, exist_ok=True)

    # decide what we're unlearning
    item_to_unlearn = config["data"]["item_to_unlearn"]

    # pull the associated base/original model
    pretrained_seed = f"seed_{config['training']['pretrained_seed']}"
    pretrained_epoch_folder = f"{config['data']['dataset']}_{config['model_class']}_{config['training']['num_epochs']}_epochs"
    print(f"pretrained seed = {pretrained_seed}, epoch folder = {pretrained_epoch_folder}")
    all_paths = glob.glob(os.path.join("./models/model_checkpoints", pretrained_seed, "pretrained", pretrained_epoch_folder, "*.pth"))
    print(all_paths)
    base_model_path = [f for f in all_paths if config["model_class"] in f][0] # janky way of only grabbing the first model checkpoint in the folder
    base_model = init_model(model_class = config["model_class"], num_classes = config['data']["num_classes"], checkpoint_path = base_model_path).to(config["device"])
    print(f"base model successfully loaded from {base_model_path}.\n")
    
    # and init a subfolder for all results pertaining to the base model
    base_subfolder = init_folder_if_not_exists( os.path.join(results_folder, "base") )
    
    # ----------------------------------------------------------------------------------- #
    # ----------------------------------------------------------------------------------- #
    # ----------------------------- DEFINE UNLEARNING LOADERS --------------------------- #
    # ----------------------------------------------------------------------------------- #
    # ----------------------------------------------------------------------------------- #
    
    # ... announce what we're unlearning
    unlearn_name = f"{config['unlearning_type']}_{item_to_unlearn}"
    print("-"*15 + "    " + "Forget set: " + unlearn_name + "\n")
    

    # ... be intelligent about setting `class_to_replace` or `percent_to_replace` if either is None
    class_param = item_to_unlearn if config['unlearning_type'] == "class" else None
    percent_param = item_to_unlearn if config['unlearning_type'] == "percent" else None
    

    # ...  ------------- get some unlearning data for this experiment ------------------- #
    # ... the dataSET is fixed across runs, and the randomness within runs is handled by simply shuffling the data loader. There is no need to actually apply the micro-seed

    # test is marked here, so we have to unmark them downstream
    marked_train_loader, _, marked_test_loader = load_dataloaders_for_experiment(
        name = config["data"]["dataset"],
        batch_size=config["data"]["batch_size"], 
        num_workers=config["data"]["num_workers"], 
        seed = config["GRAND_SEED"], 
        class_to_replace=class_param, 
        percent_to_replace=percent_param, 
        only_mark=True,
        val=False
        )
    # we make sure forget and retain sets are shuffled, to allow randomness across runs
    print("Training - forget vs retain split:")
    forget_loader, retain_loader = split_forget_retain(marked_train_loader, batch_size=config["data"]["batch_size"], shuffle = True, num_workers=config["data"]["num_workers"])
    
    
    # for datasets we're just evaling on, want shuffle = False
    print("Split 20 percent of `retain` for the MIAs...")
    retain_one_loader, retain_two_loader = split_random(retain_loader, p = .2, seed = config["GRAND_SEED"], batch_size=config["data"]["batch_size"], shuffle = False, num_workers=config["data"]["num_workers"])
    
    # unmark the test set
    unmark_dataset(marked_test_loader.dataset)
    
    unlearning_loaders = {
        "forget": forget_loader, # forget is always taken from train
        "retain": retain_loader,
        "test": marked_test_loader, # this is the FULL test set (now no longer marked)
        "retain_one": retain_one_loader, # This is passed as the TRAINING data to the MIA
        "retain_two": retain_two_loader # this is the TEST-TRAIN data for the MIA (to gut check that it indeed predicts "member" for these
    }

    # evaluate how good your base model is on this particular forget set
    if config["measure_base_results"]:

        print("---------- Evaluating metrics on base model...\n")        
        
        base_name = f"base_{unlearn_name}"
        base_results, base_out = measure_solo_metrics(
            model = base_model, 
            dataloaders = unlearning_loaders, 
            device = config["device"]
            )
        base_results["type"] = "base"
        
        # ... save base results and pth out
        with open(os.path.join(base_subfolder, f"{base_name}.json"), "w") as f:
            json.dump(base_results, f, indent=4)
        base_out_path = os.path.join(base_subfolder, f"{base_name}_out.pth")
        torch.save(base_out, base_out_path)
    else:
        # might still need base_out_path
        base_name = f"base_{unlearn_name}"
        base_out_path = os.path.join(base_subfolder, f"{base_name}_out.pth")


    # evaluate retrained from scratch models on this scenario
    if config["measure_retrain_results"]:
        
        print("---------- Evaluating metrics on retrain models...\n")
        
        # confirm results subfolder
        retrain_subfolder = init_folder_if_not_exists( os.path.join(results_folder, "retrain") )

        # find model checkpoints
        # --- this nesting is gross but works for now
        retrain_seed = f"seed_{ config['training']['retrained_from_scratch_seeds'][ config['unlearning_type'] ] }"
        print(f"retrain_seed = {retrain_seed}\n")
        retrain_checkpoints = glob.glob(os.path.join("./models/model_checkpoints", retrain_seed, "retrain_from_scratch", "*.pth"))
        print(f"retrain_checkpoints: {retrain_checkpoints}\n")
        
        # NEED TO ENSURE RETRAIN REFERENCE IS CONSISTENT
        # for each retrained model in the relevant checkpoint folder ...
        for i, ch in enumerate(retrain_checkpoints, start = 1):
            
            # ... pull the model
            retrain_model = init_model(
                model_class = config["model_class"], 
                num_classes = config['data']["num_classes"], 
                checkpoint_path = ch,
                ).to(config["device"])
            
            # ... set a name and measure stuff
            retrain_name = f"retrain_run_{i}_{unlearn_name}"
            retrain_results, retrain_out = measure_solo_metrics(
                model = retrain_model, 
                dataloaders = unlearning_loaders, 
                device = config["device"]
                )
            retrain_results["type"] = "retrain"

            # ... and save results
            with open(os.path.join(retrain_subfolder, f"{retrain_name}.json"), "w") as f:
                json.dump(retrain_results, f, indent=4)
            
            retrain_out_path = os.path.join(retrain_subfolder, f"{retrain_name}_out.pth")
            torch.save(retrain_out, retrain_out_path)
        print(f"Using retrain_out.pth file from {retrain_out_path}")
    else:
        # if not measuring this run,
        # just pull the first one as reference
        retrain_subfolder = os.path.join(results_folder, "retrain")
        all_paths = glob.glob(os.path.join(retrain_subfolder, "*.pth"))
        retrain_out_path = [f for f in all_paths][-1]
        print(f"NOT measuring retrain results this time...")
        print(f"Using retrain_out.pth file from {retrain_out_path}\n")
        # retrain_out_path = os.path.join(retrain_subfolder, f"{retrain_name}_out.pth")

    # ----------------------------------------------------------------------------------- #
    # ----------------------------------------------------------------------------------- #
    # ------------------------------- DO SOME UNLEARNING -------------------------------- #
    # ----------------------------------------------------------------------------------- #
    # ----------------------------------------------------------------------------------- #

    print("-"*54)
    print("-"*15 + "  " + f"BEGINNING UNLEARNING" + "  " + "-"*15)
    print("-"*54 + "\n")
    
    # ... THEN, for each unlearning method, 
    for m, method in enumerate(config["unlearning"]["methods"], start = 1):
    
        # ... do a bunch of runs, where ...
        for i in range(1, config["num_runs"]+1):

            run_seed = config["GRAND_SEED"] * 10_000 * m + i
            setup_seed(run_seed)

            # ... open new wandb session per method (so that data for all runs is stored in one session)
            wandb.init(
                project="Verifying-Unlearning-2026",
                name=f"{config['GRAND_SEED']}_{method}_{unlearn_name}_run_{i}",
                config=config,
                reinit= "finish_previous"
                )
                
            print("="*25 + "    " + f"RUN {i}\n")

            # ----------------------------------------------------------------------------------- #
            # ----------------------------------------------------------------------------------- #
            # ----------------------- DO A BUNCH OF UNLEARNING METHODS -------------------------- #
            # ----------------------------------------------------------------------------------- #
            # ----------------------------------------------------------------------------------- #
                
            # ... we need a new copy of the base model to begin unlearning each method on.
            # Instead of deepcopy:
            unlearn_model = init_model(model_class=config["model_class"], num_classes = config['data']["num_classes"], checkpoint_path = None).to(config["device"]) # specify "None" in that it is empty, not pretrained
            unlearn_model.load_state_dict(base_model.state_dict()) # we do this to avoid the overhead of deepcopying the model before every run

            # ... has to be in eval mode I think (so BarchNorm layers aren't screwed)
            unlearn_model.eval()
            
            # ... actually doing the unlearning (results are written and saved out underneath this function)
            _ = do_unlearning(
                base_results_folder = f"{results_folder}/unlearn/run_{i}",
                
                method_hyperparams = config["unlearning"][method],
                device = config["device"],

                method = method, # here, it is a string, and is converted to a function underneath
                model = unlearn_model,
                dataloaders = dict(unlearning_loaders), # shallow copy: prevents methods from clobbering each other's loaders
                run = i,
                forget_set_type = config['unlearning_type'],
                unlearning_item = item_to_unlearn,
                w_and_b = True,
                checkpoint_subfolder = checkpoint_subfolder,

                # we add a blank model, just in case we need it for bad_teacher or SCRUB
                blank_model = init_model(model_class=config["model_class"], num_classes = config['data']["num_classes"], checkpoint_path = None).to(config["device"]),
                seed = run_seed,

                # this function needs to be aware of where `retrain_out` pth's are saved
                retrain_out_path = retrain_out_path # by default, we just use the most recent retrain out (might need to loop through all of them later)

                )
            
        # this closes the unlearning method wandb session
        wandb.finish()


    print("-"*70)
    print("-"*19 + "  " + f'FINISHED EXPERIMENT, SEED {config["GRAND_SEED"]}' + "  " + "-"*19)
    print("-"*70 + "\n")
    

### Check metrics on unlearned models

In [5]:
# MAKE A RANDOM SEED
exp_config["GRAND_SEED"] = 4

# DO EXP
run_experiment(
    config = exp_config, 
    results_folder = f"results/seed_{exp_config['GRAND_SEED']}", 
    checkpoint_folder="models/model_checkpoints"
    )

===================  RUNNING EXPERIMENT, SEED 4  ===================

setup random seed = 4
All models will be of class ResNet.

pretrained seed = seed_4, epoch folder = CIFAR10_ResNet_100_epochs
['./models/model_checkpoints/seed_4/pretrained/CIFAR10_ResNet_100_epochs/ResNet_1.pth', './models/model_checkpoints/seed_4/pretrained/CIFAR10_ResNet_100_epochs/ResNet_2.pth', './models/model_checkpoints/seed_4/pretrained/CIFAR10_ResNet_100_epochs/ResNet_3.pth']
The normalize layer is contained in the network
base model successfully loaded from ./models/model_checkpoints/seed_4/pretrained/CIFAR10_ResNet_100_epochs/ResNet_1.pth.

---------------    Forget set: class_5

========== DATALOADER INFO
Dataset: CIFAR-10
Train: 50000 images for training
Test: 10000 images for testing
Replaced class 5 in train
Training augmentation = randomcrop(32,4) + randomhorizontalflip + colorjitter + randomrotation + normalize
Validation/Test augmentation = normalize
num_workers = 4


Training - forget vs retain spl

=========================    RUN 1

The normalize layer is contained in the network
The normalize layer is contained in the network
Executing unlearning with FT...

---------- Epoch 1

Epoch: [1][7/88]	Loss 0.0027 (0.0048)	Accuracy 100.000 (99.829)	Time 1.34
Epoch: [1][15/88]	Loss 0.0024 (0.0058)	Accuracy 100.000 (99.829)	Time 0.81
Epoch: [1][23/88]	Loss 0.0012 (0.0060)	Accuracy 100.000 (99.797)	Time 0.81
Epoch: [1][31/88]	Loss 0.0013 (0.0060)	Accuracy 100.000 (99.823)	Time 0.81
Epoch: [1][39/88]	Loss 0.0127 (0.0062)	Accuracy 99.609 (99.819)	Time 0.81
Epoch: [1][47/88]	Loss 0.0082 (0.0063)	Accuracy 99.805 (99.817)	Time 0.81
Epoch: [1][55/88]	Loss 0.0058 (0.0058)	Accuracy 99.805 (99.836)	Time 0.81
Epoch: [1][63/88]	Loss 0.0063 (0.0055)	Accuracy 99.805 (99.847)	Time 0.81
Epoch: [1][71/88]	Loss 0.0042 (0.0054)	Accuracy 99.805 (99.851)	Time 0.81
Epoch: [1][79/88]	Loss 0.0022 (0.0054)	Accuracy 100.000 (99.854)	Time 0.81
Epoch: [1][87/88]	Loss 0.0135 (0.0055)	Accuracy 99.342 (99.844)	Time 0.

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


ToW,▁█
epoch,▁█
epoch_duration,█▁
run,▁▁
total_unlearning_time_up_to_now,▁█
train_acc,▆▆██▆████▆▅██▆█▅▄██▆████▆▅███▅▆█▆▆▃██▁▆▆
train_acc_avg,▄▅▇▆▆▇▄██▇▆▅▆▁▂▆▆▆▇▇▆▆▅▃▆▆▆▇▇▇█▆██▇▃▅▅▅▅
train_loss,▁▃▃▂▁▂▂▂▂▂▂▂█▃▂▂▃▂▂▂▂▁▃▄▂▃▂▂▃▇▃▁▃▁▂▂▅▁▂▁
train_loss_avg,▄▆▅▆▆▂▃▃▃▃▂▃▂▂▃▃█▇▆▅▄▄▄▁▃▃▂▄▄▂▂▂▁▂▂▂▂▅▄▄
unlearning_item,▁▁
ToW,0.05859


=========================    RUN 2

The normalize layer is contained in the network
The normalize layer is contained in the network
Executing unlearning with FT...

results/seed_4/unlearn/run_2/FT doesn't exist - creating it...

---------- Epoch 1

Epoch: [1][7/88]	Loss 0.0071 (0.0041)	Accuracy 99.805 (99.927)	Time 1.25
Epoch: [1][15/88]	Loss 0.0194 (0.0060)	Accuracy 99.609 (99.866)	Time 0.81
Epoch: [1][23/88]	Loss 0.0080 (0.0065)	Accuracy 99.609 (99.829)	Time 0.81
Epoch: [1][31/88]	Loss 0.0044 (0.0063)	Accuracy 100.000 (99.829)	Time 0.81
Epoch: [1][39/88]	Loss 0.0042 (0.0061)	Accuracy 99.805 (99.829)	Time 0.80
Epoch: [1][47/88]	Loss 0.0023 (0.0060)	Accuracy 99.805 (99.821)	Time 0.81
Epoch: [1][55/88]	Loss 0.0103 (0.0063)	Accuracy 99.609 (99.815)	Time 0.81
Epoch: [1][63/88]	Loss 0.0092 (0.0063)	Accuracy 99.609 (99.802)	Time 0.81
Epoch: [1][71/88]	Loss 0.0141 (0.0062)	Accuracy 99.414 (99.810)	Time 0.81
Epoch: [1][79/88]	Loss 0.0083 (0.0063)	Accuracy 99.609 (99.807)	Time 0.81
Epoch: [1][

...

ToW,▁█
epoch,▁█
epoch_duration,█▁
impair_loss,█▂▁▁
impair_loss_avg,█▂▁▁
run,▁▁
total_unlearning_time_up_to_now,▁█
train_acc,▁▂▃▃▄▄▄▄▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇████████████
train_acc_avg,▁▁▁▂▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇███████████████
train_loss,█▇▇▆▆▅▅▅▅▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▁▂▂▂▂▂▁▁▁▁▁▁▁▁▁
+2,...


=========================    RUN 3

The normalize layer is contained in the network
The normalize layer is contained in the network
Executing unlearning with UNSIR...

Training UNSIR noise...

Epoch: [1] 	 Loss: 292.70611572265625
Epoch: [2] 	 Loss: 245.39309692382812
Epoch: [3] 	 Loss: 203.83212280273438
Epoch: [4] 	 Loss: 167.71542358398438
Epoch: [5] 	 Loss: 136.5909423828125
Total samples: 14120
Noise samples: 5120
Retain samples: 9000

results/seed_4/unlearn/run_3/UNSIR doesn't exist - creating it...

---------- Epoch 1

Performing impair step...

Epoch: [1][0/28]	[impair]	Loss 5.4253 (5.4253)	Accuracy 56.250 (56.250)	Time 0.36
Epoch: [1][8/28]	[impair]	Loss 1.5320 (2.4396)	Accuracy 45.312 (42.426)	Time 1.66
Epoch: [1][16/28]	[impair]	Loss 1.3428 (1.9311)	Accuracy 50.195 (45.485)	Time 1.67
Epoch: [1][24/28]	[impair]	Loss 1.3693 (1.7413)	Accuracy 46.875 (46.305)	Time 1.67
Performing repair step...

Epoch: [1][0/88]	[repair]	Loss 3.8461 (3.8461)	Accuracy 12.891 (12.891)	Time 0.57
Ep

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


ToW,▁█
epoch,▁█
epoch_duration,█▁
impair_loss,█▁▁▁
impair_loss_avg,█▂▁▁
run,▁▁
total_unlearning_time_up_to_now,▁█
train_acc,▁▃▃▃▄▄▅▅▅▅▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇███▇███▇███████
train_acc_avg,▁▂▂▂▂▂▄▄▄▄▅▅▅▅▅▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇██████████
train_loss,█▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+2,...


----------------------------------------------------------------------
-------------------  FINISHED EXPERIMENT, SEED 4  -------------------
----------------------------------------------------------------------

